In [7]:
import pandas as pd
import numpy as np

# Import all three models we will be comparing
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [8]:
# Load the preprocessed data we saved in notebook 02
# Load training features - includes synthetic SMOTE patients
x_train = pd.read_csv("../outputs/processed_data/X_train_resampled.csv")

# Load training target - includes synthetic SMOTE labels
y_train = pd.read_csv("../outputs/processed_data/y_train_resampled.csv").squeeze()

# Load test features - real patients only, never modified
x_test = pd.read_csv("../outputs/processed_data/X_test.csv")

# Load test target - real patient outcomes only
y_test = pd.read_csv("../outputs/processed_data/y_test.csv").squeeze()

print("=== DATA LOADED SUCCESSFULLY ===")
print("Training and test sets are ready for model training")
print()
print("Training set size (after SMOTE):")
print(x_train.shape)
print()
print("Test set size (real patients only):")
print(x_test.shape)

=== DATA LOADED SUCCESSFULLY ===
Training and test sets are ready for model training

Training set size (after SMOTE):
(5452, 17)

Test set size (real patients only):
(805, 17)


In [9]:
# ============================================================
# MODEL 1: LOGISTIC REGRESSION (with scaled data)
# ============================================================

from sklearn.preprocessing import StandardScaler

# Create scaler and fit on training data only
scaler = StandardScaler()

# fit_transform learns statistics from training data and scales it
x_train_scaled = scaler.fit_transform(x_train)

# transform applies the same scaling to test data without learning from it
x_test_scaled = scaler.transform(x_test)

print("=== DATA SCALING COMPLETE ===")
print("All features now have mean=0 and standard deviation=1")

# Create a Logistic Regression model object
lr_model = LogisticRegression(
    max_iter=1000,              # Maximum number of iterations to converge
    class_weight="balanced",   # Extra penalty for misclassifying minority class
    random_state=42            # Reproducible results
)

print("=== TRAINING LOGISTIN REGRESSION ===")
print("Model is learning patterns from 5,452 training patients...")
lr_model.fit(x_train_scaled, y_train)
print("Training complete")

# Use the trained model to predict outcomes for test patients
lr_predictions = lr_model.predict(x_test_scaled)

# Predict probabilities instead of just 0 or 1
lr_probabilities = lr_model.predict_proba(x_test_scaled)[:, 1]

print()
print("LOGISTIC REGRESSION RESULTS ===")

# Accuracy: what percentage of all predictions were correct
print("Accuracy:", round(accuracy_score(y_test, lr_predictions), 4))

# F1 Score: balances precision and recall
# We use the Dead class (1) because that is the class we care about most
print("F1 Score (Dead class):", round(f1_score(y_test, lr_predictions), 4))

# ROC-AUC: how well the model separates Alive from Dead overall
print("ROC-AUC:", round(roc_auc_score(y_test, lr_probabilities), 4))

print()
print("=== FULL CLASSIFICATION REPORT ===")
print("Shows precision, recall and F1 for both Alive and Dead classes")
print(classification_report(y_test, lr_predictions, target_names=["Alive", "Dead"]))

=== DATA SCALING COMPLETE ===
All features now have mean=0 and standard deviation=1
=== TRAINING LOGISTIN REGRESSION ===
Model is learning patterns from 5,452 training patients...
Training complete

LOGISTIC REGRESSION RESULTS ===
Accuracy: 0.7354
F1 Score (Dead class): 0.3107
ROC-AUC: 0.6979

=== FULL CLASSIFICATION REPORT ===
Shows precision, recall and F1 for both Alive and Dead classes
              precision    recall  f1-score   support

       Alive       0.88      0.80      0.84       682
        Dead       0.26      0.39      0.31       123

    accuracy                           0.74       805
   macro avg       0.57      0.59      0.57       805
weighted avg       0.78      0.74      0.76       805



In [10]:
# ============================================================
# MODEL 2: RANDOM FOREST
# ============================================================

# Create a Random Forest model object
rf_model = RandomForestClassifier(
    n_estimators=100,             # Number of trees in the forest
    class_weight="balanced",      # Extra penalty for misclassifying Dead patients
    random_state=42               # Reproducible results
)

print("=== TRAINING RNADOM FOREST ===")
print("Building 100 decision trees from 5452 training patients...")
rf_model.fit(x_train, y_train)
print("Training complete.")

# Predict class labels for test patients using unscaled data
rf_predictions = rf_model.predict(x_test)

# Predict probabilities for ROC-AUC calculation
rf_probabilities = rf_model.predict_proba(x_test)[:, 1]

print()
print("=== RANDOM FOREST RESULTS ===")
print("Accuracy:", round(accuracy_score(y_test, rf_predictions), 4))
print("F1 Score (Dead class):", round(f1_score(y_test, rf_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_probabilities), 4))

print()
print("=== FULL CLASSIFICATION REPORT ===")
print("Shows precision, recall and F1 for both Alive and Dead classes")
print(classification_report(y_test, rf_predictions, target_names=["Alive", "Dead"]))

=== TRAINING RNADOM FOREST ===
Building 100 decision trees from 5452 training patients...
Training complete.

=== RANDOM FOREST RESULTS ===
Accuracy: 0.7739
F1 Score (Dead class): 0.3053
ROC-AUC: 0.6718

=== FULL CLASSIFICATION REPORT ===
Shows precision, recall and F1 for both Alive and Dead classes
              precision    recall  f1-score   support

       Alive       0.88      0.85      0.86       682
        Dead       0.29      0.33      0.31       123

    accuracy                           0.77       805
   macro avg       0.58      0.59      0.59       805
weighted avg       0.79      0.77      0.78       805



In [11]:
# ============================================================
# MODEL 3 — XGBOOST
# ============================================================

# This tells XGBoost how much extra attention to pay to Dead patients
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

print("=== SCALE POS WEIGHT ===")
print("This is the ratio of Alive to Dead patients in training set")
print("XGBoost uses this to handle class imbalance")
print(round(scale_pos_weight, 2))

xgb_model = XGBClassifier(
    n_estimators=100,                   # Number of boosting rounds
    learning_rate=0.1,                  # How much each tree corrects previous
    max_depth=6,                        # Maximum depth of each tree
    scale_pos_weight=scale_pos_weight,  # Handles class imbalance
    random_state=42,                    # Reproducible results
    eval_metric="logloss",              # Evaluation metric during training
    verbosity=0                         # Suppress training output messages
)

print()
print("=== TRAINING XGBOOST ===")
print("Training 100 boosting rounds on 5452 training patients...")
xgb_model.fit(x_train, y_train)
print("Training complete.")

# Predict class labels for test patients
xgb_predictions = xgb_model.predict(x_test)

# Predict probabilities for ROC-AUC calculation
xgb_probabilities = xgb_model.predict_proba(x_test)[:, 1]

print()
print("=== XGBOOST RESULTS ===")
print("Accuracy:", round(accuracy_score(y_test, xgb_predictions), 4))
print("F1 Score (Dead class):", round(f1_score(y_test, xgb_predictions), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, xgb_probabilities), 4))

print()
print("=== FULL CLASSIFICATION REPORT ===")
print("Shows precision, recall and F1 for both Alide and Dead classes")
print(classification_report(y_test, xgb_predictions, target_names=["Alive", "Dead"]))

=== SCALE POS WEIGHT ===
This is the ratio of Alive to Dead patients in training set
XGBoost uses this to handle class imbalance
1.0

=== TRAINING XGBOOST ===
Training 100 boosting rounds on 5452 training patients...
Training complete.

=== XGBOOST RESULTS ===
Accuracy: 0.7602
F1 Score (Dead class): 0.318
ROC-AUC: 0.6815

=== FULL CLASSIFICATION REPORT ===
Shows precision, recall and F1 for both Alide and Dead classes
              precision    recall  f1-score   support

       Alive       0.88      0.83      0.85       682
        Dead       0.28      0.37      0.32       123

    accuracy                           0.76       805
   macro avg       0.58      0.60      0.59       805
weighted avg       0.79      0.76      0.77       805



In [12]:
import joblib
import os

os.makedirs('../outputs/models', exist_ok=True)

# Save Logistic Regression model. We save both the model and the scaler together. They must always be used as a pair
joblib.dump(lr_model, '../outputs/models/logistic_regression.pkl')
joblib.dump(scaler, '../outputs/models/scaler.pkl')
print("=== LOGISTIC REGRESSION MODEL SAVED ===")
print("Saved to outputs/models/logistic_regression.pkl")

# Save Random Forest model
joblib.dump(rf_model, '../outputs/models/random_forest.pkl')
print()
print("=== RANDOM FOREST MODEL SAVED ===")
print("Saved to outputs/models/random_forest.pkl")

# Save XGBoost model
joblib.dump(xgb_model, '../outputs/models/xgboost.pkl')
print()
print("=== XGBOOST MODEL SAVED ===")
print("Saved to outputs/models/xgboost.pkl")

print()
print("=== ALL MODELS SAVED SUCCESSFULLY ===")
print("Models can be loaded in evaluation notebook without retraining")

=== LOGISTIC REGRESSION MODEL SAVED ===
Saved to outputs/models/logistic_regression.pkl

=== RANDOM FOREST MODEL SAVED ===
Saved to outputs/models/random_forest.pkl

=== XGBOOST MODEL SAVED ===
Saved to outputs/models/xgboost.pkl

=== ALL MODELS SAVED SUCCESSFULLY ===
Models can be loaded in evaluation notebook without retraining
